# ಪಾಠ 13 - ಏಜೆಂಟ್ ಸ್ಮರಣೆ


## ಸೆಟಪ್

ಈ ನೋಟ್‌ಬುಕ್ **Microsoft Agent Framework** (MAF) ಬಳಸಿ **ಸ್ಥಾಯಿಶೀಲ ಸ್ಮರಣೆ**ಯೊಂದಿಗೆ ಪ್ರವಾಸ ಬುಕ್ಕಿಂಗ್ ಏಜೆಂಟ್ ಅನ್ನು ಹೇಗೆ ನಿರ್ಮಿಸಬೇಕು ಎಂಬುದನ್ನು ತೋರಿಸುತ್ತದೆ.

ನೀವು ವಿವಿಧ ರೀತಿಯ ಏಜೆಂಟ್ ಸ್ಮರಣೆ — ಕೆಲಸ ಮಾಡುವ, ಕನಿಷ್ಟಕಾಲಿಕ, ಮತ್ತು ದೀರ್ಘಕಾಲিক — ಹೇಗೆ ಏಜೆಂಟ್ ಭೇಟಿಗಳ ನಡುವೆ ಮಾಹಿತಿಯನ್ನು ಉಳಿಸಿಕೊಂಡು ಬಳಸುತ್ತದೆ ಎಂದು ಕಲಿಯುತ್ತೀರಿ.

**ಮೂಲಭೂತ ಅಗತ್ಯಗಳು:**
- ಒಂದು ಅಜೂರ್ AI ಫೌಂಡ್ರೀ ಪ್ರಾಜೆಕ್ಟ್, ಅದು ಚಾಟ್ ಮಾದರಿಯನ್ನು ನಿಯೋಜಿಸಿದೆ (ಉದಾ: `gpt-4o-mini`).
- ಅಜೂರ್ CLI ಮೂಲಕ ಲಾಗಿನ್ ಆಗಿರುವುದು — ನಿಮ್ಮ ಟರ್ಮಿನಲ್‌ನಲ್ಲಿ `az login` ಅನ್ನು চালಿಸಿ.
- `AZURE_AI_PROJECT_ENDPOINT` — ನಿಮ್ಮ ಅಜೂರ್ AI ಫೌಂಡ್ರೀ ಪ್ರಾಜೆಕ್ಟ್ ಎಂಡ್‌ಪಾಯಿಂಟ್.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ನಿಮ್ಮ ನಿಯೋಜಿತ ಮಾದರಿಯ ಹೆಸರು.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## ಏಜೆಂಟ್ ಮೆಮರಿ ವಿಧಗಳು

AI ಏಜೆಂಟ್ಸ್ ವಿಭಿನ್ನ ರೀತಿಯ ಮೆಮರಿಯನ್ನು ಉಪಯೋಗಿಸಬಹುದು, ಪ್ರತಿಯೊಂದು ವಿಭಿನ್ನ ಉದ್ದೇಶದ ಸೇವೆ ನೀಡುತ್ತದೆ:

### ಕಾರ್ಯಾಚರಣೆ ಮೆಮರಿ  
ಸಂವಾದ ಸರಣಿ — ಒಬ್ಬ ಸೆಷನ್‌ನಲ್ಲಿ ವಿನಿಮಯಗೊಂಡ ಸಂದೇಶಗಳು. ಏಜೆಂಟ್ coherence ಕಾಯ್ದುಕೊಳಲು ಅದೇ ಸರಣಿಯ ಹಿಂದಿನ ಸಂದೇಶಗಳಿಗೆ ಹಿಂತಿರುಗಬಹುದು. MAF ನಲ್ಲಿ ಇದು **`agent.create_session()`** ಮೂಲಕ ರಚಿಸಲಾಗುತ್ತದೆ, ಇದು `AgentSession` ಅನ್ನು ಹಿಂತಿರುಗಿಸುತ್ತದೆ.

### ಅಲ್ಪಕಾಲ ಮೆಮರಿ  
ಒಂದು ಕೆಲಸ ಅಥವಾ ಸೆಷನ್ ಅವಧಿಗೆ ಉಳಿಯುವ ಮಾಹಿತಿ ಆದರೆ ಶಾಶ್ವತವಾಗಿ ಸಂಗ್ರಹಿಸಲಾಗದದ್ದು. ಉದಾಹರಣೆಗೆ, ಏಜೆಂಟ್ ಬಹು-ತಿರುವು ಯೋಜನೆ ಸಂವಾದದ ವೇಳೆ ಮಾಹಿತಿಗಳನ್ನು ಸಂಗ್ರಹಿಸಿ ಅವುಗಳನ್ನು ಅಂತಿಮ ಪ್ರಯಾಣವೃತ್ತಾಂತವನ್ನು ಉತ್ಪಾದಿಸಲು ಬಳಸಬಹುದು.

### ದೀರ್ಘಕಾಲ ಮೆಮರಿ  
**ಸೆಷನ್‌ಗಳ ನಡುವೆ** ಉಳಿಯುವ ಆಸಕ್ತಿಗಳು ಮತ್ತು ವಾಸ್ತವಗಳು. ಮರುಬರುವ ಬಳಕೆದಾರರು ತಮ್ಮ ಆಹಾರ ನಿರ್ಬಂಧಗಳು ಅಥವಾ ಪ್ರಯಾಣ ಶೈಲಿಯನ್ನು ಪುನರಾವರ್ತಿಸಲು ಆಗಬೇಕಾಗಿಲ್ಲ. ದೀರ್ಘಕಾಲ ಮೆಮರಿ ಸಾಮಾನ್ಯವಾಗಿ ಹೊರಗಿನ ಸಂಗ್ರಹಣೆ—ಡೇಟಾಬೇಸ್, ಕಡತ ಅಥವಾ ವೆಕ್ಟರ್ ಸೂಚ್ಯಂಕ—ದಿಂದ ಬೆಂಬಲಿತವಾಗಿದ್ದು, ತಂತ್ರಗಳನ್ನು ಮೂಲಕ ಏಜೆಂಟ್‌ಗೆ ಪ್ರದರ್ಶಿಸಲಾಗುತ್ತದೆ.


## ಸೆಷನ್‌ಗಳೊಂದಿಗೆ ವರ್ಕಿಂಗ್ ಮೆಮರಿ

ಸ್ಮರಣೆಯ ಸರಳ ರೂಪವು ಸಂಭಾಷಣೆ ಸೆಷನ್ ಆಗಿದೆ. ನೀವು ಒಂದೇ ಸೆಷನ್ ಅನ್ನು (agent.create_session() ಮೂಲಕ ರಚಿಸಲ್ಪಟ್ಟ) ಮರುಕಳಿಸುವ `agent.run()` ಕಾಲ್‌ಗಳಿಗೆ ಹಂಚಿದಾಗ, ಏಜೆಂಟ್ ಆ ಸಂಭಾಷಣೆಯ ಪೂರ್ಣ ಇತಿಹಾಸವನ್ನು ನೋಡಿ ಹಿಂದಿನ ವಿವರಗಳನ್ನು ನೆನಪಿಸಿಕೊಳ್ಳಬಹುದು.

ನಾವು ಒಂದು ಪ್ರವಾಸ ಏಜೆಂಟ್ ಅನ್ನು ರಚಿಸಿ ವರ್ಕಿಂಗ್ ಮೆಮರಿಯನ್ನು ಪ್ರದರ್ಶಿಸೋಣ.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ಏಜೆಂಟ್ ನಿರ್ವಹಣಾ ಬಜೆಟ್ ಅನ್ನು ಸರಿಯಾಗಿ ನೆನಪಿಸಿಕೊಂಡಿದೆ ಏಕೆಂದರೆ ಎರಡೂ ಸಂದೇಶಗಳು ಒಂದೇ ಸೆಷನ್ ಹಂಚಿಕೊಂಡಿವೆ. ಇದು **ಕೆಲಸ ಸ್ಮৃতি** — ಇದು ಸೆಷನ್ ಜೀವನಾವಧಿಗೆ ಮಾತ್ರ ಅಸ್ತಿತ್ವದಲ್ಲಿರುತ್ತದು.

### ಹೊಸ ತಂತು ಏನಾಗುತ್ತದೆ?

ನಾವು **ಹೊಸ** ಸೆಷನ್ ಸೃಷ್ಟಿಸಿದರೆ, ಏಜೆಂಟಿಗೆ ಹಿಂದಿನ ಸಂಭಾಷಣೆಯ ಬಗ್ಗೆ ಯಾವುದೇ ಸ್ಮೃತಿ ಇರುವುದಿಲ್ಲ:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## ದೀರ್ಘಕಾಲೀನ ಮೆಮೊರಿ ಮಾದರಿ

ಬಳಕೆದಾರರ ಪ್ರಥಮತೆಗಳನ್ನು **ಸೆಷನ್‌ಗಳ ಅಂತರದಲ್ಲಿ** ನೆನಪಿಡಲು, ನಾವು ಸಂವಾದ ದಾಳಿನ ಹೊರಗೆ ಇರುವ ಒಂದು ಸ್ಥಾಯಿ ಸಂಗ್ರಹಣೆಯನ್ನು ಅಗತ್ಯವಿದೆ. ಏಜೆಂಟ್ ಈ ಸಂಗ್ರಹಣೆಯನ್ನು ಪ್ರವೇಶಿಸಲು **ಟೂಲ್‌ಗಳು** — ಮಾಹಿತಿ ಸಂರಕ್ಷಿಸಲು ಮತ್ತು ಪುನಃ ಪಡೆದುಕೊಳ್ಳಲು ಕರೆಯಬಹುದಾದ ಕಾರ್ಯಗಳನ್ನು ಬಳಕೆ ಮಾಡುತ್ತದೆ.

ಕೆಳಗಿನಂತೆ ನಾವು ಸರಳ ಮೌಲ್ಯ ಸಂಗ್ರಹಣೆಯನ್ನು (ಉತ್ಪಾದನೆಯಲ್ಲಿ ನೀವು ಇದನ್ನು ಡೇಟಾಬೇಸ್ ಅಥವಾ ವೆಕ್ಟರ್ ಸೂಚ್ಯಂಕದಿಂದ ಬೆಂಬಲಿಸಬಹುದು) ಜಾರಿಗೆ ತಂದಿದ್ದು, ಏಜೆಂಟ್ ಬಳಸಿಕೊಳ್ಳಬಹುದಾದ ಟೂಲ್‌ಗಳಾಗಿ ಬಹಿರಂಗಪಡಿಸಿದ್ದೇವೆ.

### ವಾಸ್ತುಶಿಲ್ಪ  
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### ದೃಶ್ಯ 1 — ಮೊದಲಬಾರಿಗೆ ಬಳಕೆದಾರ ವಾಣಿಜ್ಯಯಾತ್ರೆಯನ್ನು ಬುಕ್ ಮಾಡುತ್ತಾನೆ

ಸರಾ ಮೊದಲ ಬಾರಿಗೆ ಭೇಟಿ ನೀಡುತ್ತಾಳೆ. ಏಜೆಂಟ್ ಅವಳ प्राथमिकತೆಗಳನ್ನು ಉಪಕರಣಗಳ ಮೂಲಕ ಸಂಗ್ರಹಿಸಿ ಅವುಗಳನ್ನು ಹೋಟೆಲ್‌ಗಳನ್ನು ಶಿಫಾರಸು ಮಾಡಲು ಬಳಸಬೇಕು.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### ಸಂದರ್ಭದಲ್ಲಿ 2 — ಸಾರಾ ವಾರಗಳ ನಂತರ ಮರಳುತ್ತದೆ

ಸಾರಾ **ಹೊಸ ಎಸೆಸನ್ನು** ಪ್ರಾರಂಭಿಸುತ್ತದೆ (ಹೊಸ ಸೆಷನ್‌ ಅನ್ನು ಅನುಕರಿಸುವುದು). ಕಾರ್ಯಸ್ಮೃತಿ ಖಾಲಿಯಾಗಿದೆ, ಆದರೆ ದೀರ್ಘಕಾಲPreference ಸಂಗ್ರಹವು ಇನ್ನೂ ಅವಳ ಮಾಹಿತಿ ಹೊಂದಿದೆ. ಏಜೆಂಟ್ ಅದನ್ನು ತಗೆದುಕೊಂಡು ವೈಯಕ್ತೀಕರಿಸಿದ ಶಿಫಾರಸುಗಳಿಗೆ ಬಳಸಬೇಕು.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಮೂರು ವಿಧದ ಏಜೆಂಟ್ ಸ್ಮರಣೆಗಳನ್ನು ಮತ್ತು ಅವುಗಳನ್ನು Microsoft Agent Framework ಮೂಲಕ ಹೇಗೆ ಅನ್ವಯಿಸಬಹುದು ಎಂಬುದನ್ನು ಕಲಿತಿರಿ:

| ಸ್ಮರಣೆ ಪ್ರಕಾರ | MAF ಯಂತ್ರವಿಧಾನ | ಆಯುಷ್ಯಾವಧಿ |
|---|---|---|
| **ಕಾರ್ಯನಿರ್ವಹಣಾ** | `agent.create_session()` | ಒಂದೇ ಸಂಭಾಷಣೆ |
| **ಕಾಲಶೀರ್ಷಕ** | ತಂತಿ ಒಳಗಿನ ಸಂಗ್ರಹಿತ ಸಂದರ್ಭ | ಒಂದೇ ಕಾರ್ಯ/ಸೆಷನ್ |
| **ದೀರ್ಘಕಾಲದ** | `@tool` ಕಾರ್ಯಗಳ ಮೂಲಕ ಪ್ರವೇಶದ ಹೊರಗಿನ ಸಂಗ್ರಹಣೆ | ಸೆಷನ್‌ಗಳ ಮಧ್ಯೆ |

### ಮುಖ್ಯ ಅಂಶಗಳು
1. **`agent.create_session()`** ಕಾರ್ಯನಿರ್ವಹಣಾ ಸ್ಮರಣೆಯನ್ನು ಒದಗಿಸುತ್ತದೆ — ಏಜೆಂಟ್ ಸೆಷನ್ ಒಳಗಿನ ಪೂರ್ಣ ಸಂಭಾಷಣಾ ಇತಿಹಾಸವನ್ನು ನೋಡಬಹುದು.
2. **ಹೊಸ ಸೆಷನ್‌ಗಳು ಸಂದರ್ಭವನ್ನು ಕಳೆದುಕೊಳ್ಳುತ್ತವೆ** — ದೀರ್ಘಕಾಲದ ಸ್ಮರಣೆಯಿಲ್ಲದೆ ಏಜೆಂಟ್ ಹಿಂದಿನ ಸಂಭಾಷಣೆಗಳನ್ನು ಸ್ಮರಿಸಲು ಸಾಧ್ಯವಿಲ್ಲ.
3. **`@tool` ಕಾರ್ಯಗಳು** ಕಂದಾಯವನ್ನು ತೃಪ್ತಿಪಡಿಸುತ್ತವೆ — ಅವು ಏಜೆಂಟಿಗೆ ಸ್ಥಿರ ಸಂಗ್ರಹಣೆಯಿಂದ ಮಾಹಿತಿ ಉಳಿಸಿ ಮತ್ತು ಪಡೆಯಲು ಅವಕಾಶ ನೀಡುತ್ತವೆ.
4. **ವೈಯಕ್ತೀಕರಣ ಕಾಲ ಕಳೆದಂತೆ మెరుగಾಗುತ್ತದೆ** — ಹೆಚ್ಚಂತೆ ಇಚ್ಛೆಗಳು ಸಂಗ್ರಹಿಸಿದಂತೆ, ಏಜೆಂಟ್ ಶಿಫಾರಸುಗಳು ಉತ್ಕೃಷ್ಟವಾಗುತ್ತವೆ.

### ನೈಜ ಜಗತ್ತಿನ ಅನ್ವಯಿಕೆಗಳು
- **ಗ್ರಾಹಕ ಸೇವೆ**: ಗ್ರಾಹಕರ ಇತಿಹಾಸ ಮತ್ತು ಇಚ್ಛೆಗಳ ನೆನಪಿಡು
- **ವೈಯುಕ್ತಿಕ ಸಹಾಯಕರು**: ದಿನಗಳು ಅಥವಾ ವಾರಗಳ ಕಾಲ ಸಂದರ್ಭವನ್ನು ಕಾಪಾಡುವುದು
- **ಆರೋಗ್ಯಸೇವೆ**: ರೋಗಿ ಮಾಹಿತಿ ಮತ್ತು ಇಚ್ಛೆಗಳ ಟ್ರಾಕಿಂಗ್
- **ಇ-ಕಾಮರ್ಸ್**: ಇತಿಹಾಸ ಆಧಾರದ ವೈಯಕ್ತೀಕೃತ ಶಾಪಿಂಗ್

### ಮುಂದಿನ ಹಂತಗಳು
- ಮೆಮೊರಿ ಡಿಕ್‌ಶನರಿಯನ್ನು ಡೇಟಾಬೇಸ್ ಅಥವಾ ವೆಕ್ಟರ್ ಸ್ಟೋರ್ (ಉದಾ: Azure AI Search) ಗೆ ಬದಲಿಸಬೇಕು
- ಸಮಯ ಸಂವೇದಿ ಮಾಹಿತಿಗಾಗಿ ಸ್ಮರಣಾಕಾಲಾವಧಿ ಸೇರಿಸಬೇಕು
- ಹಂಚಿಕೊಳ್ಳಲಾದ ಸ್ಮರಣೆಯುಳ್ಳ ಬಹು-ಏಜೆಂಟ್ ವ್ಯವಸ್ಥೆಗಳು ನಿರ್ಮಿಸಬೇಕು
- ಜ್ಞಾನ-ಗ್ರಾಫ್ ಬೆಂಬಲಿತ ಸ್ಮರಣೆಯಗಾಗಿ Cognee ನೋಟ್‌ಬುಕ್ ಅನ್ನು ಅನ್ವೇಷಿಸಬೇಕು


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಬಾಹ್ಯಸೂಚನೆ**:  
ಈ ಪತ್ರಿಕೆಯನ್ನು ಎಐ ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ಸಮಷ್ಠಿಯತ್ತ ಪ್ರಯತ್ನಿಸಿದರೂ ಕೂಡ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸತ್ಯತೆಗಳಾಗಿರಬಲ್ಲವು ಎಂದು ಗಮನಿಸಿ. ಮೂಲ ಭಾಷೆಯಲ್ಲಿನ ಮೂಲ ಪತ್ರಿಕೆ ಪ್ರಾಮಾಣಿಕ ಮೂಲ ಎಂದು ಪರಿಗಣಿಸಬೇಕು. ಮಹತ್ವದ ಮಾಹಿತಿಗಾಗಿ ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದದಿಂದ ಉಂಟಾಗುವ ಯಾವುದೇ ಅರ್ಥಮೈಮಾಸಿಕೆ или ತಪ್ಪು ನಿರ್ವಹಣೆಗೆ ನಾವು ಜವಾಬ್ದಾರರಾಗುವುದಿಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
